# Example: Population Trends

We are now ready to work with large DataFrames of data. The file below contains "Annual Estimates of the Resident Population by Single Year of Age and Sex for the United States." Notice that `pl.read_csv` can read data directly from a URL.

In [1]:
import polars as pl
path_data = '../../../assets/data/'

In [2]:
# As of August 2021, this census file is online here:
data = 'http://www2.census.gov/programs-surveys/popest/technical-documentation/file-layouts/2010-2019/nc-est2019-agesex-res.csv'

# A local copy can be accessed here in case census.gov moves the file:
# data = path_data + 'nc-est2019-agesex-res.csv'

full_census_table = pl.read_csv(data)
full_census_table

SEX,AGE,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,POPESTIMATE2011,POPESTIMATE2012,POPESTIMATE2013,POPESTIMATE2014,POPESTIMATE2015,POPESTIMATE2016,POPESTIMATE2017,POPESTIMATE2018,POPESTIMATE2019
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,0,3944153,3944160,3951430,3963092,3926570,3931258,3954787,3983981,3954773,3893990,3815343,3783052
0,1,3978070,3978090,3957730,3966225,3977549,3942698,3948891,3973133,4002903,3972711,3908830,3829599
0,2,4096929,4096939,4090621,3970654,3978925,3991740,3958711,3966321,3991349,4020045,3987032,3922044
0,3,4119040,4119051,4111688,4101644,3981531,3991017,4005928,3974351,3982984,4006946,4033038,3998665
0,4,4063170,4063186,4077346,4121488,4111490,3992502,4004032,4020292,3989750,3997280,4018719,4043323
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,97,53582,53605,54165,57123,59221,60670,62407,68608,69709,76276,82100,84358
2,98,36641,36675,37567,40078,42561,43866,45887,46611,52122,52978,57593,62607
2,99,26193,26214,26098,26995,29076,30672,32245,33560,34278,38489,39576,42528


Only the first five and last five rows of the DataFrame are displayed. Later we will see how to display the entire DataFrame; however, this is typically not useful with large DataFrames.

A [description of the data](https://www2.census.gov/programs-surveys/popest/technical-documentation/file-layouts/2010-2019/nc-est2019-agesex-res.pdf) appears online.

The `SEX` column contains numeric codes: `0` stands for the total, `1` for male, and `2` for female. The assumptions underlying this binary categorization are questionable. We will discuss them in the next section.

In this section we will focus on age and population size.

The `AGE` column contains ages in completed years, but the special value `999` is a sum of the total population. The "age" `100` too has a special status. In that category, the Census Bureau includes all people aged 100 or more.

The other columns contain estimates of the US population in each category of sex and age in the years 2010 through 2019. The Census is decennial: it takes place every 10 years. The most recent Census was held in 2020 and the one before that in 2010. The Census Bureau also estimates population changes each year. As explained in the Bureau's description of its [methodology](https://www2.census.gov/programs-surveys/popest/technical-documentation/methodology/2010-2020/methods-statement-v2020-final.pdf), it "adds [the estimated changes] to the last decennial census to produce updated population estimates every year."

Typically, a publicly available DataFrame will contain more information than necessary for a particular investigation or analysis. To get the large DataFrame into a more usable form, we have to do some *data cleaning*.

Suppose we are only interested in the population changes from 2014 to 2019. Let's `select` the relevant columns.

In [3]:
partial_census_table = full_census_table.select('SEX', 'AGE', 'POPESTIMATE2014', 'POPESTIMATE2019')
partial_census_table

SEX,AGE,POPESTIMATE2014,POPESTIMATE2019
i64,i64,i64,i64
0,0,3954787,3783052
0,1,3948891,3829599
0,2,3958711,3922044
0,3,4005928,3998665
0,4,4004032,4043323
…,…,…,…
2,97,62407,84358
2,98,45887,62607
2,99,32245,42528


We can simplify the labels of the selected columns.

In [4]:
us_pop = partial_census_table.rename({"POPESTIMATE2014": "2014", "POPESTIMATE2019": "2019"})
us_pop

SEX,AGE,2014,2019
i64,i64,i64,i64
0,0,3954787,3783052
0,1,3948891,3829599
0,2,3958711,3922044
0,3,4005928,3998665
0,4,4004032,4043323
…,…,…,…
2,97,62407,84358
2,98,45887,62607
2,99,32245,42528


## Ages 97-100
As a warm-up, let's examine the total population, labeled by `SEX` code 0. Since all these rows will have the same value 0 in the `SEX` column, we will drop that column.

In [5]:
us_pop_by_age = us_pop.filter(pl.col("SEX") == 0).drop("SEX")
us_pop_by_age

AGE,2014,2019
i64,i64,i64
0,3954787,3783052
1,3948891,3829599
2,3958711,3922044
3,4005928,3998665
4,4004032,4043323
…,…,…
97,82948,116969
98,59546,86150
99,41277,57124


Now let's look at the population in the highest ages. `closed="left"` includes the lower endpoint and excludes the upper one, so ages 97, 98, 99, and 100 are kept and 101 is not.

In [6]:
us_pop_by_age.filter(pl.col("AGE").is_between(97, 101, closed="left"))

AGE,2014,2019
i64,i64,i64
97,82948,116969
98,59546,86150
99,41277,57124
100,70685,100322


Not surprisingly, the numbers of people are smaller at higher ages. For example, there are fewer 99-year-olds than 98-year-olds. 

But the numbers for `AGE` 100 are quite a bit larger than those for age 99. That is because the row with `AGE` 100 doesn't just represent 100-year-olds. It also includes those who are older than 100. 

## Percent Change

Each column of the DataFrame `us_pop_by_age` is a Polars Series of the same length, and so columns can be combined using arithmetic. The Polars Series below contains the change in population between 2014 and 2019. There is one entry corresponding to each row of `us_pop_by_age`.

In [1]:
# Option 1: Assign with alias so it is ready for with_columns (gives a named Series)
change = (us_pop_by_age.get_column("2019") - us_pop_by_age.get_column("2014")).alias("Change")
change

Change
i64
-171735
-119292
-36667
-7263
39291
…
34021
26604
15847


In [2]:
# Option 2: Simple difference (keeps the name of the left Series)
change_simple = us_pop_by_age.get_column("2019") - us_pop_by_age.get_column("2014")
change_simple

2019
i64
-171735
-119292
-36667
-7263
39291
…
34021
26604
15847


We can augment `us_pop_by_age` with a column that contains these changes, both in absolute terms and as percents relative to the value in 2014.

In [8]:
us_pop_change = us_pop_by_age.with_columns(
    change,
    (change / us_pop_by_age.get_column("2014")).alias("Percent Change"),
)
us_pop_change = us_pop_change.with_columns(
    ((pl.col("Percent Change") * 100).round(2).cast(pl.String) + "%").alias("Percent Change")
)
us_pop_change

AGE,2014,2019,Change,Percent Change
i64,i64,i64,i64,str
0,3954787,3783052,-171735,"""-4.34%"""
1,3948891,3829599,-119292,"""-3.02%"""
2,3958711,3922044,-36667,"""-0.93%"""
3,4005928,3998665,-7263,"""-0.18%"""
4,4004032,4043323,39291,"""0.98%"""
…,…,…,…,…
97,82948,116969,34021,"""41.01%"""
98,59546,86150,26604,"""44.68%"""
99,41277,57124,15847,"""38.39%"""


The first rows of the `Percent Change` column are negative, demonstrating a drop in population at the youngest ages. However, the overall population grew by about 9.9 million people, a percent change of just over 3%.

In [9]:
us_pop_change.filter(pl.col("AGE") == 999)

AGE,2014,2019,Change,Percent Change
i64,i64,i64,i64,str
999,318301008,328239523,9938515,"""3.12%"""


Let us compare this to the change at each age. For ease of interpretation, we will sort the DataFrame in decreasing order of the absolute change in population, contained in the column `Change`.

In [10]:
us_pop_change.filter(
    pl.col("AGE") < 999
).sort("Change", descending=True)

AGE,2014,2019,Change,Percent Change
i64,i64,i64,i64,str
72,2191642,3191048,999406,"""45.6%"""
68,2567511,3345475,777964,"""30.3%"""
69,2530460,3252423,721963,"""28.53%"""
70,2461426,3136704,675278,"""27.43%"""
71,2516392,3083083,566691,"""22.52%"""
…,…,…,…,…
44,4375450,3967275,-408175,"""-9.33%"""
53,4523245,4068851,-454394,"""-10.05%"""
43,4319265,3854040,-465225,"""-10.77%"""


Take a look at the top few rows. While the percent change is about 3% for the overall population, it jumps to well over 20% for the people in their late sixties and early seventies. This stunning change contributes to what is known as the greying of America.

What could explain this large increase? We can explore this question by examining the years in which the relevant groups were born.

- Those who were in the age group 69 to 72 in 2014 were born in the years 1942 to 1945. The attack on Pearl Harbor was in late 1941, and by 1942 U.S. forces were heavily engaged in a massive war that ended in 1945. 

- Those who were 69 to 72 years old in 2019 were born in the years 1947 to 1950, at the height of the post-WWII baby boom in the United States. 

The post-war jump in births is a major reason for the large changes that we have observed.